# DeepShield — Wavelet-CLIP Training (Kaggle)
Train the WaveletCLIP branch on Kaggle T4/P100 GPU.

**Setup before running:**
1. Mount datasets: `deepfake-detection-challenge` (DFDC), your private FF++ dataset, your private Celeb-DF v2 dataset
2. Replace `YOUR_USERNAME` in Cell 2 with your GitHub username
3. Run all cells in order
4. Download checkpoint from Cell 12

## Cell 1 — Install dependencies

In [ ]:
!pip install timm==1.0.3 transformers==4.41.0 insightface==0.7.3 PyWavelets==1.6.0 pytorch-grad-cam==1.5.0 wandb==0.17.0 albumentations==1.4.7 -q

## Cell 2 — Clone repo

In [ ]:
import os
os.chdir('/kaggle/working')
!git clone https://github.com/YOUR_USERNAME/deepshield.git
os.chdir('/kaggle/working/deepshield')
import sys
sys.path.insert(0, '/kaggle/working/deepshield')
print('Repo cloned and added to path')

## Cell 3 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Enable GPU in Kaggle settings (Settings > Accelerator > GPU T4 x2 or P100)'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

## Cell 4 — Check disk space + set paths

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Disk: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB')

PATHS = {
    'dfdc':      '/kaggle/input/deepfake-detection-challenge',
    'ff':        '/kaggle/input/faceforensics-pp',       # your private dataset
    'celeb_df':  '/kaggle/input/celeb-df-v2',            # your private dataset
    'faces_out': '/kaggle/working/faces',
    'ckpt_dir':  '/kaggle/working/checkpoints/wavelet_clip',
}

import os
for name, path in PATHS.items():
    exists = os.path.exists(path)
    print(f'  {name:10s}: {path}  [{"OK" if exists else "NOT FOUND"}]')

os.makedirs(PATHS['faces_out'] + '/train/real', exist_ok=True)
os.makedirs(PATHS['faces_out'] + '/train/fake', exist_ok=True)
os.makedirs(PATHS['faces_out'] + '/val/real',   exist_ok=True)
os.makedirs(PATHS['faces_out'] + '/val/fake',   exist_ok=True)
os.makedirs(PATHS['ckpt_dir'],                  exist_ok=True)

## Cell 5 — Face extraction (memory-safe, video by video)

In [ ]:
import shutil
from pathlib import Path
from models.retinaface_detector import FaceDetector
from data.preprocess.extract_faces import extract_from_video, extract_from_image, SUPPORTED_VIDEO, SUPPORTED_IMAGE

detector = FaceDetector(device='cuda')
MAX_DFDC_VIDEOS = 3000
MIN_FREE_GB = 3.0

def check_space():
    _, _, free = shutil.disk_usage('/kaggle/working')
    return free / 1e9

print(f'Starting extraction. Free: {check_space():.1f} GB')

# ── FF++ real sequences → train/real ──
ff_real_dir = Path(PATHS['ff']) / 'original_sequences'
if ff_real_dir.exists():
    vids = sorted(ff_real_dir.rglob('*.mp4'))[:500]  # cap at 500
    print(f'Extracting {len(vids)} FF++ real videos...')
    for i, v in enumerate(vids):
        if check_space() < MIN_FREE_GB: print('Space low, stopping'); break
        extract_from_video(v, Path(PATHS['faces_out']) / 'train' / 'real', detector, fps_sample=1.0)
        if i % 50 == 0: print(f'  [{i}/{len(vids)}] Free: {check_space():.1f} GB')
else:
    print(f'FF++ not found at {ff_real_dir} — skipping')

# ── FF++ fake sequences → train/fake ──
for method in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
    ff_fake_dir = Path(PATHS['ff']) / 'manipulated_sequences' / method
    if not ff_fake_dir.exists(): continue
    vids = sorted(ff_fake_dir.rglob('*.mp4'))[:250]
    print(f'Extracting {len(vids)} FF++ {method} fake videos...')
    for i, v in enumerate(vids):
        if check_space() < MIN_FREE_GB: print('Space low, stopping'); break
        extract_from_video(v, Path(PATHS['faces_out']) / 'train' / 'fake', detector, fps_sample=1.0)

# ── DFDC → train/fake ──
dfdc_dir = Path(PATHS['dfdc'])
if dfdc_dir.exists():
    all_dfdc = sorted(dfdc_dir.rglob('*.mp4'))
    print(f'Found {len(all_dfdc)} DFDC videos. Processing up to {MAX_DFDC_VIDEOS}...')
    for i, v in enumerate(all_dfdc):
        if i >= MAX_DFDC_VIDEOS: break
        if check_space() < MIN_FREE_GB: print(f'Space low at video {i}, stopping'); break
        extract_from_video(v, Path(PATHS['faces_out']) / 'train' / 'fake', detector, fps_sample=0.5)
        if i % 100 == 0: print(f'  [{i}/{MAX_DFDC_VIDEOS}] Free: {check_space():.1f} GB')
else:
    print(f'DFDC not found at {dfdc_dir}')

# ── Celeb-DF v2 → val split ──
celeb_dir = Path(PATHS['celeb_df'])
if celeb_dir.exists():
    celeb_real = sorted((celeb_dir / 'Celeb-real' / 'videos').rglob('*.mp4'))[:100]
    celeb_fake = sorted((celeb_dir / 'Celeb-synthesis' / 'videos').rglob('*.mp4'))[:100]
    print(f'Extracting Celeb-DF val: {len(celeb_real)} real, {len(celeb_fake)} fake...')
    for v in celeb_real:
        extract_from_video(v, Path(PATHS['faces_out']) / 'val' / 'real', detector, fps_sample=1.0)
    for v in celeb_fake:
        extract_from_video(v, Path(PATHS['faces_out']) / 'val' / 'fake', detector, fps_sample=1.0)
else:
    print(f'Celeb-DF not found at {celeb_dir}')

# Count
for split in ['train', 'val']:
    for label in ['real', 'fake']:
        d = Path(PATHS['faces_out']) / split / label
        n = len(list(d.rglob('*.jpg')))
        print(f'{split}/{label}: {n} crops')
print(f'Final free disk: {check_space():.1f} GB')

## Cell 6 — Config

In [ ]:
import torch
CFG = {
    'img_size':           224,
    'batch_size':         32,          # T4 16GB with fp16
    'epochs':             40,
    'lr_wavelet':         2e-4,
    'lr_clip_proj':       5e-5,
    'weight_decay':       1e-4,
    'mixed_precision':    True,
    'num_workers':        2,
    'grad_accum_steps':   1,
    'focal_loss_gamma':   2.0,
    'focal_loss_alpha':   0.25,
    'grad_clip':          1.0,
    'seed':               42,
    'ckpt_dir':           PATHS['ckpt_dir'],
    'face_dir':           PATHS['faces_out'],
}
print('Config:', CFG)

## Cell 7 — Datasets

In [ ]:
import random, numpy as np
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])

from data.preprocess.prepare_datasets import WaveletCLIPDataset
from torch.utils.data import DataLoader

train_ds = WaveletCLIPDataset(CFG['face_dir'], split='train', img_size=CFG['img_size'])
val_ds   = WaveletCLIPDataset(CFG['face_dir'], split='val',   img_size=CFG['img_size'])
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  drop_last=True,  num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,                  num_workers=CFG['num_workers'], pin_memory=True)

## Cell 8 — Model init

In [ ]:
from models.wavelet_clip import WaveletCLIP

device = torch.device('cuda')
print('Loading CLIP (downloads ~1.7 GB on first run)...')
model = WaveletCLIP().to(device)
print('Model loaded.')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M / {total/1e6:.1f}M total')

## Cell 9 — Optimizer (separate param groups)

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.wavelet_branch.parameters(),       'lr': CFG['lr_wavelet']},
    {'params': model.clip_projection.proj.parameters(), 'lr': CFG['lr_clip_proj']},
    {'params': model.fusion.parameters(),               'lr': CFG['lr_wavelet']},
], weight_decay=CFG['weight_decay'])

total_steps = len(train_loader) * CFG['epochs']
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[CFG['lr_wavelet'], CFG['lr_clip_proj'], CFG['lr_wavelet']],
    total_steps=total_steps,
)
print(f'Optimizer ready. Total steps: {total_steps}')

## Cell 10 — Focal Loss

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt    = torch.exp(-bce)
        focal = self.alpha * (1.0 - pt) ** self.gamma * bce
        return focal.float().mean()  # .float() prevents NaN with fp16

criterion = FocalLoss(gamma=CFG['focal_loss_gamma'], alpha=CFG['focal_loss_alpha'])
scaler    = torch.amp.GradScaler('cuda', enabled=CFG['mixed_precision'])
print('Loss and scaler ready.')

## Cell 11 — Training loop

In [ ]:
import json
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def validate_epoch(model, loader):
    model.eval()
    all_scores, all_labels, total_loss = [], [], 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_scores.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels.cpu())
    scores = torch.cat(all_scores).numpy().flatten()
    labels = torch.cat(all_labels).numpy().flatten()
    auc = roc_auc_score(labels, scores) if len(set(labels)) > 1 else 0.5
    return total_loss / len(loader), auc

best_auc = 0.0
history  = []
Path(CFG['ckpt_dir']).mkdir(parents=True, exist_ok=True)

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}', leave=False)):
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss   = criterion(logits, labels) / CFG['grad_accum_steps']
        scaler.scale(loss).backward()
        if (step + 1) % CFG['grad_accum_steps'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        train_loss += loss.item() * CFG['grad_accum_steps']

    train_loss /= len(train_loader)
    val_loss, val_auc = validate_epoch(model, val_loader)
    lr = optimizer.param_groups[0]['lr']

    record = {'epoch': epoch, 'train_loss': round(train_loss,4), 'val_auc': round(val_auc,4), 'lr': round(lr,8)}
    history.append(record)
    print(f'Epoch {epoch:3d}/{CFG["epochs"]} | loss={train_loss:.4f} | val_auc={val_auc:.4f} | lr={lr:.2e}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_auc': val_auc},
                   f'{CFG["ckpt_dir"]}/wavelet_clip_best_auc.pth')
        print(f'  ★ New best AUC: {best_auc:.4f}')

    # Disk check every 5 epochs
    if epoch % 5 == 0:
        import shutil
        _, _, free = shutil.disk_usage('/kaggle/working')
        print(f'  Disk free: {free/1e9:.1f} GB')

with open(f'{CFG["ckpt_dir"]}/history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Best val AUC: {best_auc:.4f}')
print(f'Checkpoint: {CFG["ckpt_dir"]}/wavelet_clip_best_auc.pth')

## Cell 12 — Download checkpoint

In [ ]:
from IPython.display import FileLink, display
import os

ckpt_path = f'{CFG["ckpt_dir"]}/wavelet_clip_best_auc.pth'
size_mb   = os.path.getsize(ckpt_path) / 1e6
print(f'Checkpoint size: {size_mb:.0f} MB')
print(f'Click to download:')
display(FileLink(ckpt_path))

# Also show training history
import json
history_path = f'{CFG["ckpt_dir"]}/history.json'
with open(history_path) as f:
    hist = json.load(f)
print(f'\nFinal 5 epochs:')
for r in hist[-5:]:
    print(f'  Epoch {r["epoch"]}: train_loss={r["train_loss"]}  val_auc={r["val_auc"]}')